In [3]:
# =========================================================
# CELL 1. IMPORT THƯ VIỆN VÀ CẤU HÌNH
# =========================================================
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

# Bản cân bằng tốc độ / chất lượng
N_ITER = 20
TOP_K = 5
MAX_CV_SPLITS = 4
N_JOBS = -1

BASE_DIR = Path.cwd()

ARTIFACT_DIR = BASE_DIR / "artifacts_logistic_regression"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = BASE_DIR / "saved_models"
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "Clause"
TARGET_COL = "code"

In [4]:
# =========================================================
# CELL 2. ĐỌC DỮ LIỆU
# =========================================================
def read_csv_robust(path):
    last_error = None
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Không đọc được file CSV: {path}\nLỗi cuối cùng: {last_error}")

# SỬA LẠI ĐƯỜNG DẪN FILE Ở ĐÂY
DATA_PATH = r"C:\Users\Admin\Downloads\annotated_clauses_topics.csv"

df = read_csv_robust(DATA_PATH)

print("=== SHAPE DỮ LIỆU ===")
print(df.shape)

print("\n=== CÁC CỘT ===")
print(df.columns.tolist())

print("\n=== 5 DÒNG ĐẦU ===")
print(df.head())

=== SHAPE DỮ LIỆU ===
(23173, 10)

=== CÁC CỘT ===
['review_id', 'Username', 'Address', 'Rating', 'Language', 'ID_clause', 'Clause', 'topic', 'code', 'topic_name']

=== 5 DÒNG ĐẦU ===
   review_id        Username                     Address  Rating Language  \
0      34501       Hà Nguyễn  Lake of the Restored Sword       4       vi   
1       7678    Stanley Noel               Cat Ba Island       4       en   
2      22893  Marine Marques               Hoa Lo Prison       5       en   
3      33263          MrPhyl             Japanese Bridge       1       en   
4       4538           JEK W            Ben Thanh Market       2       it   

  ID_clause                                             Clause  topic code  \
0   34501.a                     The lake is as green as paint.      0   EL   
1    7678.d  but the 12km hike (there and back) was well wo...      0   EL   
2   22893.b                               It was quite crowded      8  TCr   
3   33263.e  but they actually physically

In [5]:
# =========================================================
# CELL 3. EDA CƠ BẢN
# =========================================================
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATED ROWS TOÀN BỘ ===")
print(df.duplicated().sum())

print(f"\n=== PHÂN BỐ NHÃN THEO {TARGET_COL} ===")
print(df[TARGET_COL].value_counts())

print(f"\n=== TỶ LỆ NHÃN THEO {TARGET_COL} (%) ===")
print((df[TARGET_COL].value_counts(normalize=True) * 100).round(2))

if "topic_name" in df.columns:
    print("\n=== THỐNG KÊ topic_name ===")
    print(df["topic_name"].value_counts())

=== MISSING VALUES ===
review_id     0
Username      0
Address       0
Rating        0
Language      0
ID_clause     0
Clause        0
topic         0
code          0
topic_name    0
dtype: int64

=== DUPLICATED ROWS TOÀN BỘ ===
0

=== PHÂN BỐ NHÃN THEO code ===
code
VE     9017
TC     1872
LC     1734
RA     1697
HL     1477
CT     1475
SE     1411
UH     1102
TCr     984
CA      898
EL      698
EC      460
VR      348
Name: count, dtype: int64

=== TỶ LỆ NHÃN THEO code (%) ===
code
VE     38.91
TC      8.08
LC      7.48
RA      7.32
HL      6.37
CT      6.37
SE      6.09
UH      4.76
TCr     4.25
CA      3.88
EL      3.01
EC      1.99
VR      1.50
Name: proportion, dtype: float64

=== THỐNG KÊ topic_name ===
topic_name
Visit Evaluation            9017
Travel Costs                1872
Local Consumption           1734
Recreational Activities     1697
Historical Learning         1477
Coastal Tourism             1475
Service Experience          1411
Urban Heritage              1102
Touri

In [6]:
# =========================================================
# CELL 4. LÀM SẠCH DỮ LIỆU
# =========================================================
required_cols = [TEXT_COL, TARGET_COL]
missing_required_cols = [col for col in required_cols if col not in df.columns]

if missing_required_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_required_cols}")

before_rows = len(df)

df = df.dropna(subset=[TEXT_COL, TARGET_COL]).copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

df = df[df[TEXT_COL] != ""].copy()

# Xóa trùng theo cặp text-label để giảm leakage và giảm thời gian train
df = df.drop_duplicates(subset=[TEXT_COL, TARGET_COL]).reset_index(drop=True)

after_rows = len(df)

print("=== SAU LÀM SẠCH ===")
print("Số dòng trước:", before_rows)
print("Số dòng sau  :", after_rows)
print("Số dòng bị loại:", before_rows - after_rows)

print("\n=== SỐ CLASS ===")
print(df[TARGET_COL].nunique())

print("\n=== CLASS COUNTS ===")
print(df[TARGET_COL].value_counts())

=== SAU LÀM SẠCH ===
Số dòng trước: 23173
Số dòng sau  : 23114
Số dòng bị loại: 59

=== SỐ CLASS ===
13

=== CLASS COUNTS ===
code
VE     8989
TC     1871
LC     1718
RA     1696
HL     1477
CT     1475
SE     1405
UH     1101
TCr     982
CA      896
EL      696
EC      460
VR      348
Name: count, dtype: int64


In [7]:
# =========================================================
# CELL 5. CHIA DỮ LIỆU TRAIN / VALID / TEST
# =========================================================
X = df[TEXT_COL]
y = df[TARGET_COL]

# Test = 10%
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=RANDOM_STATE
)

# Valid = 10% toàn bộ dataset
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=1/9,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

labels = sorted(pd.unique(df[TARGET_COL]))

print("=== KÍCH THƯỚC DỮ LIỆU ===")
print(f"Train: {len(X_train)} ({len(X_train)/len(df):.2%})")
print(f"Valid: {len(X_valid)} ({len(X_valid)/len(df):.2%})")
print(f"Test : {len(X_test)} ({len(X_test)/len(df):.2%})")

=== KÍCH THƯỚC DỮ LIỆU ===
Train: 18490 (79.99%)
Valid: 2312 (10.00%)
Test : 2312 (10.00%)


In [8]:
# =========================================================
# CELL 6. TẠO PIPELINE
# =========================================================
def build_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            strip_accents="unicode",
            sublinear_tf=True,
            dtype=np.float32
        )),
        ("clf", LogisticRegression(random_state=RANDOM_STATE))
    ])

In [9]:
# =========================================================
# CELL 7. KHÔNG GIAN THAM SỐ TUNING
# =========================================================
param_distributions = [
    {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [1, 2, 3, 5],
        "tfidf__max_df": [0.90, 0.95, 0.98],
        "tfidf__stop_words": [None, "english"],
        "tfidf__max_features": [20000, 40000, 60000, None],
        "clf__solver": ["lbfgs"],
        "clf__penalty": ["l2"],
        "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
        "clf__class_weight": [None, "balanced"],
        "clf__max_iter": [2000, 3000],
    },
    {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [1, 2, 3, 5],
        "tfidf__max_df": [0.90, 0.95, 0.98],
        "tfidf__stop_words": [None, "english"],
        "tfidf__max_features": [20000, 40000],
        "clf__solver": ["saga"],
        "clf__penalty": ["l1", "l2"],
        "clf__C": [0.1, 0.3, 1.0, 3.0],
        "clf__class_weight": [None, "balanced"],
        "clf__max_iter": [2500, 3500],
    }
]

print("Đã tạo xong không gian tham số để tune.")

Đã tạo xong không gian tham số để tune.


In [10]:
# =========================================================
# CELL 8. RANDOMIZED SEARCH CV
# =========================================================
min_class_count_train = y_train.value_counts().min()
n_splits = min(MAX_CV_SPLITS, int(min_class_count_train))
n_splits = max(2, n_splits)

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=RANDOM_STATE
)

random_search = RandomizedSearchCV(
    estimator=build_pipeline(),
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="f1_macro",
    n_jobs=N_JOBS,
    cv=cv,
    verbose=2,
    random_state=RANDOM_STATE,
    return_train_score=False,
    refit=False,
    error_score="raise"
)

random_search.fit(X_train, y_train)

search_results = pd.DataFrame(random_search.cv_results_).sort_values(
    ["rank_test_score", "mean_test_score"],
    ascending=[True, False]
).reset_index(drop=True)

search_results["cv_stability_score"] = (
    search_results["mean_test_score"] - search_results["std_test_score"]
)

search_results.to_csv(ARTIFACT_DIR / "random_search_full_results.csv", index=False)

print("=== TOP 10 CV RESULTS ===")
print(
    search_results[
        ["rank_test_score", "mean_test_score", "std_test_score", "cv_stability_score", "params"]
    ].head(10)
)

Fitting 4 folds for each of 20 candidates, totalling 80 fits
=== TOP 10 CV RESULTS ===
   rank_test_score  mean_test_score  std_test_score  cv_stability_score  \
0                1         0.879822        0.006541            0.873282   
1                2         0.834212        0.003536            0.830676   
2                3         0.833864        0.003711            0.830153   
3                4         0.832261        0.008637            0.823624   
4                5         0.821742        0.008491            0.813251   
5                6         0.798948        0.007038            0.791910   
6                7         0.798235        0.005943            0.792292   
7                7         0.798235        0.005943            0.792292   
8                9         0.783450        0.004967            0.778483   
9               10         0.779086        0.005493            0.773593   

                                              params  
0  {'tfidf__stop_words': 'englis

In [11]:
# =========================================================
# CELL 9. HÀM TÍNH METRICS
# =========================================================
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
    }

print("Đã tạo hàm compute_metrics.")

Đã tạo hàm compute_metrics.


In [12]:
# =========================================================
# CELL 10. CHỌN TOP CANDIDATE TRÊN VALID SET
# =========================================================
top_k = min(TOP_K, len(search_results))
candidate_rows = []

for i in range(top_k):
    params = search_results.loc[i, "params"]

    model = build_pipeline()
    model.set_params(**params)
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)

    train_metrics = compute_metrics(y_train, train_pred)
    valid_metrics = compute_metrics(y_valid, valid_pred)

    candidate_rows.append({
        "candidate_id": i + 1,
        "cv_rank": search_results.loc[i, "rank_test_score"],
        "cv_mean_valid_f1_macro": search_results.loc[i, "mean_test_score"],
        "cv_std_valid_f1_macro": search_results.loc[i, "std_test_score"],
        "cv_stability_score": search_results.loc[i, "cv_stability_score"],
        "train_accuracy": train_metrics["accuracy"],
        "train_f1_macro": train_metrics["f1_macro"],
        "valid_accuracy": valid_metrics["accuracy"],
        "valid_f1_macro": valid_metrics["f1_macro"],
        "valid_precision_macro": valid_metrics["precision_macro"],
        "valid_recall_macro": valid_metrics["recall_macro"],
        "gap_train_valid_f1": train_metrics["f1_macro"] - valid_metrics["f1_macro"],
        "params": params,
        "params_str": json.dumps(params, ensure_ascii=False, default=str),
    })

candidate_df = pd.DataFrame(candidate_rows).sort_values(
    ["valid_f1_macro", "gap_train_valid_f1", "cv_stability_score"],
    ascending=[False, True, False]
).reset_index(drop=True)

best_valid_f1 = candidate_df["valid_f1_macro"].max()

robust_pool = candidate_df[
    candidate_df["valid_f1_macro"] >= (best_valid_f1 - 0.003)
].copy()

robust_pool = robust_pool.sort_values(
    ["gap_train_valid_f1", "cv_stability_score", "valid_accuracy"],
    ascending=[True, False, False]
).reset_index(drop=True)

best_params = robust_pool.loc[0, "params"]

candidate_df.drop(columns=["params"]).to_csv(
    ARTIFACT_DIR / "candidate_validation_results.csv",
    index=False
)

print("=== CANDIDATE RESULTS ===")
print(
    candidate_df[
        [
            "candidate_id",
            "cv_rank",
            "cv_mean_valid_f1_macro",
            "train_f1_macro",
            "valid_f1_macro",
            "gap_train_valid_f1",
            "params_str"
        ]
    ]
)

print("\n=== BEST PARAMS MỚI ===")
print(json.dumps(best_params, ensure_ascii=False, indent=2, default=str))

=== CANDIDATE RESULTS ===
   candidate_id  cv_rank  cv_mean_valid_f1_macro  train_f1_macro  \
0             1        1                0.879822        0.938909   
1             3        3                0.833864        0.873602   
2             4        4                0.832261        0.977337   
3             2        2                0.834212        0.867560   
4             5        5                0.821742        0.967436   

   valid_f1_macro  gap_train_valid_f1  \
0        0.895854            0.043055   
1        0.856838            0.016764   
2        0.852508            0.124829   
3        0.848365            0.019194   
4        0.837427            0.130010   

                                          params_str  
0  {"tfidf__stop_words": "english", "tfidf__ngram...  
1  {"tfidf__stop_words": null, "tfidf__ngram_rang...  
2  {"tfidf__stop_words": null, "tfidf__ngram_rang...  
3  {"tfidf__stop_words": null, "tfidf__ngram_rang...  
4  {"tfidf__stop_words": "english", "tfidf_

In [13]:
# =========================================================
# CELL 11. FIT MODEL ĐÁNH GIÁ TRÊN TRAIN
# =========================================================
eval_model = build_pipeline()
eval_model.set_params(**best_params)
eval_model.fit(X_train, y_train)

print("Đã fit eval_model với best_params mới.")

Đã fit eval_model với best_params mới.


In [14]:
# =========================================================
# CELL 12. HÀM ĐÁNH GIÁ CHI TIẾT
# =========================================================
def evaluate_split(model, X_data, y_true, split_name, labels):
    y_pred = model.predict(X_data)

    metrics = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels]
    )

    print(f"\n=== ĐÁNH GIÁ TRÊN {split_name.upper()} ===")
    print(f"Accuracy            : {metrics['accuracy']:.4f}")
    print(f"Precision (macro)   : {metrics['precision_macro']:.4f}")
    print(f"Recall (macro)      : {metrics['recall_macro']:.4f}")
    print(f"F1-score (macro)    : {metrics['f1_macro']:.4f}")
    print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
    print(f"Recall (weighted)   : {metrics['recall_weighted']:.4f}")
    print(f"F1-score (weighted) : {metrics['f1_weighted']:.4f}")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=4))

    report_df.to_csv(ARTIFACT_DIR / f"classification_report_{split_name}.csv", index=True)
    cm_df.to_csv(ARTIFACT_DIR / f"confusion_matrix_{split_name}.csv", index=True)

    return metrics, report_df, cm_df

In [15]:
# =========================================================
# CELL 13. ĐÁNH GIÁ TRAIN / VALID / TEST
# =========================================================
train_metrics, train_report_df, train_cm_df = evaluate_split(
    eval_model, X_train, y_train, "train", labels
)

valid_metrics, valid_report_df, valid_cm_df = evaluate_split(
    eval_model, X_valid, y_valid, "valid", labels
)

test_metrics, test_report_df, test_cm_df = evaluate_split(
    eval_model, X_test, y_test, "test", labels
)


=== ĐÁNH GIÁ TRÊN TRAIN ===
Accuracy            : 0.9557
Precision (macro)   : 0.9525
Recall (macro)      : 0.9274
F1-score (macro)    : 0.9389
Precision (weighted): 0.9557
Recall (weighted)   : 0.9557
F1-score (weighted) : 0.9552

Classification report:
              precision    recall  f1-score   support

          CA     0.9272    0.9596    0.9431       717
          CT     0.9541    0.9678    0.9609      1180
          EC     0.9798    0.9239    0.9510       368
          EL     0.9350    0.9551    0.9449       557
          HL     0.9605    0.9255    0.9426      1181
          LC     0.9503    0.9607    0.9555      1374
          RA     0.9436    0.9012    0.9219      1356
          SE     0.9720    0.9573    0.9646      1124
          TC     0.9676    0.9165    0.9413      1497
         TCr     0.9782    0.9148    0.9454       786
          UH     0.9513    0.9535    0.9524       881
          VE     0.9566    0.9905    0.9733      7191
          VR     0.9062    0.7302    0.80

In [16]:
# =========================================================
# CELL 14. KIỂM TRA OVERFITTING
# =========================================================
train_valid_gap = train_metrics["f1_macro"] - valid_metrics["f1_macro"]
train_test_gap = train_metrics["f1_macro"] - test_metrics["f1_macro"]
valid_test_gap = abs(valid_metrics["f1_macro"] - test_metrics["f1_macro"])

print("=== KIỂM TRA OVERFITTING ===")
print(f"Train macro-F1 : {train_metrics['f1_macro']:.4f}")
print(f"Valid macro-F1 : {valid_metrics['f1_macro']:.4f}")
print(f"Test macro-F1  : {test_metrics['f1_macro']:.4f}")
print(f"Gap train-valid: {train_valid_gap:.4f}")
print(f"Gap train-test : {train_test_gap:.4f}")
print(f"Gap valid-test : {valid_test_gap:.4f}")

if train_valid_gap > 0.05:
    print("CẢNH BÁO: Có dấu hiệu overfitting vì train-valid gap > 0.05")
else:
    print("OK: Chưa thấy overfitting mạnh giữa train và valid")

if valid_test_gap > 0.05:
    print("LƯU Ý: Valid và test lệch nhau khá nhiều")
else:
    print("OK: Valid và test khá đồng nhất")

=== KIỂM TRA OVERFITTING ===
Train macro-F1 : 0.9389
Valid macro-F1 : 0.8959
Test macro-F1  : 0.8922
Gap train-valid: 0.0431
Gap train-test : 0.0467
Gap valid-test : 0.0036
OK: Chưa thấy overfitting mạnh giữa train và valid
OK: Valid và test khá đồng nhất


In [17]:
# =========================================================
# CELL 15. TRAIN FINAL MODEL TRÊN TRAIN + VALID
# =========================================================
X_final = pd.concat([X_train, X_valid], axis=0)
y_final = pd.concat([y_train, y_valid], axis=0)

final_model = build_pipeline()
final_model.set_params(**best_params)
final_model.fit(X_final, y_final)

print("Đã train final_model trên train + valid.")

Đã train final_model trên train + valid.


In [18]:
# =========================================================
# CELL 16. LƯU FULL PIPELINE
# =========================================================
joblib.dump(final_model, ARTIFACT_DIR / "logistic_topic_classifier.joblib")

with open(ARTIFACT_DIR / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, ensure_ascii=False, indent=2, default=str)

save_info = {
    "text_column": TEXT_COL,
    "target_column": TARGET_COL,
    "model_file": "logistic_topic_classifier.joblib",
    "best_params_file": "best_params.json",
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "test_size": len(X_test),
    "labels": list(labels),
    "n_iter_random_search": N_ITER,
    "top_k_candidates": TOP_K,
    "cv_splits": n_splits,
    "search_note": "Balanced speed-quality tuning on new dataset"
}

with open(ARTIFACT_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2, default=str)

print("Đã lưu full pipeline.")
print("Model:", ARTIFACT_DIR / "logistic_topic_classifier.joblib")

Đã lưu full pipeline.
Model: c:\Users\Admin\Downloads\artifacts_logistic_regression\logistic_topic_classifier.joblib


In [19]:
# =========================================================
# CELL 17. TEST LOAD MODEL VÀ DỰ ĐOÁN
# =========================================================
loaded_model = joblib.load(ARTIFACT_DIR / "logistic_topic_classifier.joblib")

new_clauses = [
    "The staff were friendly and helpful",
    "The ticket price was too expensive",
    "The scenery was very beautiful and peaceful"
]

predictions = loaded_model.predict(new_clauses)

result = pd.DataFrame({
    "Clause": new_clauses,
    "Predicted_code": predictions
})

print(result)

                                        Clause Predicted_code
0          The staff were friendly and helpful             SE
1           The ticket price was too expensive             TC
2  The scenery was very beautiful and peaceful             EL


In [20]:
# =========================================================
# CELL 18. LƯU RIÊNG MODEL + VECTORIZER + CONFIG
# =========================================================
lr_vectorizer = final_model.named_steps["tfidf"]
lr_model = final_model.named_steps["clf"]

joblib.dump(lr_model, SAVED_MODELS_DIR / "lr_model.pkl")
joblib.dump(lr_vectorizer, SAVED_MODELS_DIR / "lr_vectorizer.pkl")

lr_config = {
    "model_name": "LogisticRegression",
    "text_column": TEXT_COL,
    "target_column": TARGET_COL,
    "labels": list(labels),
    "random_state": RANDOM_STATE,
    "n_iter_random_search": N_ITER,
    "top_k_candidates": TOP_K,
    "cv_splits": n_splits,
    "best_params": best_params,
    "files": {
        "model_file": "lr_model.pkl",
        "vectorizer_file": "lr_vectorizer.pkl"
    }
}

with open(SAVED_MODELS_DIR / "lr_config.json", "w", encoding="utf-8") as f:
    json.dump(lr_config, f, ensure_ascii=False, indent=2, default=str)

print("=== ĐÃ LƯU DẠNG TÁCH RIÊNG ===")
print("Model      :", SAVED_MODELS_DIR / "lr_model.pkl")
print("Vectorizer :", SAVED_MODELS_DIR / "lr_vectorizer.pkl")
print("Config     :", SAVED_MODELS_DIR / "lr_config.json")

=== ĐÃ LƯU DẠNG TÁCH RIÊNG ===
Model      : c:\Users\Admin\Downloads\saved_models\lr_model.pkl
Vectorizer : c:\Users\Admin\Downloads\saved_models\lr_vectorizer.pkl
Config     : c:\Users\Admin\Downloads\saved_models\lr_config.json
